# 🚦 Monitor de Radares Ativos — MG (INMETRO/RBMLQ)
Busca radares verificados nos últimos **N dias**, geocodifica via **vGeo MG** (com fallback Nominatim) e salva uma **planilha formatada no Google Drive**.

> **Fonte:** `https://servicos.rbmlq.gov.br/dados-abertos/mg/medidores.json`

## ⚙️ 1. Configurações

In [ ]:
# ── Filtros ───────────────────────────────────────────────
DIAS_RECENTES    = 7
SITUACOES_ATIVAS = ["aprovado", "ativo", "apto"]

# ── Google Drive ─────────────────────────────────────────
NOME_PLANILHA = "Radares_Ativos_MG"   # nome do .xlsx
PASTA_DRIVE   = ""                    # subpasta no Drive (vazio = raiz)

# ── Geocodificação ────────────────────────────────────────
VGEO_BASE_URL           = "https://vgeo.mg.gov.br/arcgis/rest/services"
USAR_NOMINATIM_FALLBACK = True

print("✅ Configurações OK")

✅ Configurações OK


## 📦 2. Dependências

In [ ]:
!pip install -q requests openpyxl geopy pandas tqdm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\andre\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 📂 3. Montar Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive"
DRIVE_OUT  = os.path.join(DRIVE_BASE, PASTA_DRIVE) if PASTA_DRIVE else DRIVE_BASE
os.makedirs(DRIVE_OUT, exist_ok=True)

PLANILHA_PATH = os.path.join(DRIVE_OUT, f"{NOME_PLANILHA}.xlsx")
print(f"📁 Destino: {PLANILHA_PATH}")

ModuleNotFoundError: No module named 'google.colab'

## 🔍 4. Buscar e filtrar radares

In [ ]:
# ── Célula 4 — substituir a célula de busca ──────────────────────────────────

import requests
import re
import pandas as pd
from datetime import datetime, timedelta
import xml.etree.ElementTree as ET

URL_JSON = "https://servicos.rbmlq.gov.br/dados-abertos/mg/medidores.json"
URL_XML  = "https://servicos.rbmlq.gov.br/dados-abertos/mg/medidores.xml"

# Headers que imitam um navegador real — evita bloqueio por User-Agent
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "application/json, text/xml, */*",
    "Accept-Language": "pt-BR,pt;q=0.9",
    "Referer":         "https://servicos.rbmlq.gov.br/",
}

# ── Tentativa 1: JSON ─────────────────────────────────────────────────────────
registros = None
try:
    print(f"⏳ Tentando JSON: {URL_JSON}")
    resp = requests.get(URL_JSON, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    raw = resp.json()

    if isinstance(raw, list):
        registros = raw
    else:
        for chave in ["medidores","data","instrumentos","registros","results"]:
            if chave in raw:
                registros = raw[chave]
                break
        else:
            registros = list(raw.values())[0]

    print(f"✅ JSON OK — {len(registros)} registros")

except Exception as e_json:
    print(f"⚠️  JSON falhou: {e_json}")

    # ── Tentativa 2: XML ─────────────────────────────────────────────────────
    try:
        print(f"\n⏳ Tentando XML: {URL_XML}")
        resp2 = requests.get(URL_XML, headers=HEADERS, timeout=30)
        resp2.raise_for_status()
        root = ET.fromstring(resp2.content)

        # Detecta tags de registro automaticamente
        # Estrutura típica: <medidores><medidor>...</medidor></medidores>
        # ou <root><item>...</item></root>
        filhos = list(root)
        if filhos:
            tag_registro = filhos[0].tag
            registros = []
            for elem in root.findall(tag_registro):
                registro = {}
                for child in elem:
                    registro[child.tag] = child.text or ""
                # Se o elemento tiver atributos
                registro.update(elem.attrib)
                registros.append(registro)

        print(f"✅ XML OK — {len(registros)} registros")

    except Exception as e_xml:
        print(f"❌ XML também falhou: {e_xml}")
        raise RuntimeError(
            "Não foi possível acessar a API RBMLQ.\n"
            "O servidor pode estar fora do ar ou bloqueando IPs do Colab.\n"
            "Tente novamente em alguns minutos ou acesse pelo navegador:\n"
            f"  {URL_JSON}"
        )

if registros:
    print(f"\n📊 Total de registros : {len(registros)}")
    print(f"🔑 Campos detectados  : {list(registros[0].keys())}")

In [ ]:
# ── Helpers ──────────────────────────────────────────────
def campo(reg, *nomes, default=""):
    for n in nomes:
        alvo = n.lower().replace("_","").replace(" ","")
        for k, v in reg.items():
            if k.lower().replace("_","").replace(" ","") == alvo:
                return str(v) if v is not None else default
    return default

def parse_data(txt):
    for fmt in ["%d/%m/%Y","%Y-%m-%d","%d-%m-%Y","%Y/%m/%d","%d/%m/%y"]:
        try:
            return datetime.strptime(txt.strip(), fmt)
        except Exception:
            continue
    return None

def extrair_km(txt):
    m = re.search(r'km\s*[:]?\s*([\d]+[.,]?[\d]*)', txt, re.IGNORECASE)
    return float(m.group(1).replace(",",".")) if m else None

def extrair_rodovia(txt):
    m = re.search(
        r'((?:BR|MG|ES|SP|RJ|GO|MS|RS|SC|PR|BA|CE|PE|PA|AM|MT|TO|PI|MA|RN|PB|AL|SE|RO|RR|AC|AP|DF)[\s-]?\d{2,3})',
        txt, re.IGNORECASE
    )
    return m.group(1).upper().replace(" ","-") if m else None

# ── Filtro ───────────────────────────────────────────────
corte = datetime.now() - timedelta(days=DIAS_RECENTES)
radares = []

for r in registros:
    situacao = campo(r, "situacao","status","resultado").lower()
    data_str = campo(r, "dataverificacao","data_verificacao","dataverif","datainicio","data")
    data_dt  = parse_data(data_str)

    if any(s in situacao for s in SITUACOES_ATIVAS) and data_dt and data_dt >= corte:
        local = campo(r, "local","localizacao","endereco")
        rod   = campo(r, "rodovia","via","br") or local
        radares.append({
            "numero":           campo(r, "numero","numeroserie","serie","id"),
            "municipio":        campo(r, "municipio","cidade","localidade"),
            "local":            local,
            "rodovia":          extrair_rodovia(rod+" "+local) or rod,
            "km":               extrair_km(local),
            "situacao":         campo(r, "situacao","status"),
            "data_verificacao": data_str,
            "data_validade":    campo(r, "datavalidade","data_validade","validade","datafim"),
            "latitude":         None,
            "longitude":        None,
            "fonte_geo":        None,
            "maps_link":        None,
        })

print(f"\n✅ Radares ativos (últimos {DIAS_RECENTES} dias): {len(radares)}")

## 🌍 5. Geocodificação — vGeo MG + Nominatim (fallback)

In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm

nom = Nominatim(user_agent="radar_mg_monitor")
geocode_nom = RateLimiter(nom.geocode, min_delay_seconds=1.1)

def vgeo(rodovia, km):
    if not rodovia or km is None:
        return None, None
    try:
        # Endpoint 1 – referência linear (LRS)
        r = requests.get(
            f"{VGEO_BASE_URL}/Rodovias/Rodovias_MG/MapServer/exts/LRSServer/networkLayers/0/geometryAtMeasure",
            params={"routeId": rodovia.replace("-", ""), "measure": km, "f": "json"},
            timeout=10
        ).json()
        g = r.get("geometry", {})
        if g.get("x"):
            return round(g["y"], 6), round(g["x"], 6)

        # Endpoint 2 – geocoder de endereço
        r2 = requests.get(
            f"{VGEO_BASE_URL}/Geocoding/Geocoder/GeocodeServer/findAddressCandidates",
            params={"SingleLine": f"{rodovia} KM {km} Minas Gerais", "outFields": "*", "f": "json"},
            timeout=10
        ).json()
        cands = r2.get("candidates", [])
        if cands:
            loc = cands[0]["location"]
            return round(loc["y"], 6), round(loc["x"], 6)
    except Exception:
        pass
    return None, None

def nominatim_geo(municipio, rodovia, km):
    try:
        if rodovia and km:
            loc = geocode_nom(f"{rodovia} km {km}, {municipio}, Minas Gerais, Brasil", country_codes="br")
            if loc:
                return round(loc.latitude, 6), round(loc.longitude, 6)
        loc2 = geocode_nom(f"{municipio}, Minas Gerais, Brasil", country_codes="br")
        if loc2:
            return round(loc2.latitude, 6), round(loc2.longitude, 6)
    except Exception:
        pass
    return None, None

print("🌍 Geocodificando...")
for radar in tqdm(radares):
    lat, lon = vgeo(radar["rodovia"], radar["km"])
    fonte = "vGeo MG"
    if lat is None and USAR_NOMINATIM_FALLBACK:
        lat, lon = nominatim_geo(radar["municipio"], radar["rodovia"], radar["km"])
        fonte = "Nominatim (OSM)"
    radar["latitude"] = lat
    radar["longitude"] = lon
    radar["fonte_geo"] = fonte if lat else "Não encontrado"
    if lat and lon:
        radar["maps_link"] = f"https://www.google.com/maps?q={lat},{lon}"

geo_ok = sum(1 for r in radares if r["latitude"])
print(f"✅ Com coordenadas: {geo_ok}/{len(radares)}")

## 📊 6. Visualizar no Colab

In [ ]:
df = pd.DataFrame(radares)
if df.empty:
    print(f"ℹ️ Nenhum radar ativo nos últimos {DIAS_RECENTES} dias.")
else:
    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_colwidth", 55)
    display(df)

## 💾 7. Salvar planilha no Google Drive

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

if df.empty:
    print("ℹ️ Nada para salvar.")
else:
    COLUNAS  = ["Número", "Município", "Local", "Rodovia", "KM", "Situação",
                "Dt. Verificação", "Validade", "Latitude", "Longitude", "Fonte Geo", "Google Maps"]
    CAMPOS   = ["numero", "municipio", "local", "rodovia", "km", "situacao",
                "data_verificacao", "data_validade", "latitude", "longitude", "fonte_geo", "maps_link"]
    LARGURAS = [12, 22, 42, 12, 8, 12, 16, 14, 12, 12, 16, 12]

    hd_fill  = PatternFill("solid", fgColor="1F4E79")
    par_fill = PatternFill("solid", fgColor="D6E4F0")
    borda    = Border(left=Side(style="thin"), right=Side(style="thin"),
                      top=Side(style="thin"), bottom=Side(style="thin"))

    wb = Workbook()
    ws = wb.active
    ws.title = "Radares Ativos"

    # Cabeçalho
    for ci, nome in enumerate(COLUNAS, 1):
        c = ws.cell(row=1, column=ci, value=nome)
        c.font = Font(bold=True, color="FFFFFF", size=11)
        c.fill = hd_fill
        c.alignment = Alignment(horizontal="center", vertical="center")
        c.border = borda
    ws.row_dimensions[1].height = 22

    # Dados
    for ri, radar in enumerate(radares, 2):
        fill = par_fill if ri % 2 == 0 else None
        for ci, fld in enumerate(CAMPOS, 1):
            valor = radar.get(fld, "")
            c = ws.cell(row=ri, column=ci, value=valor)
            c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            c.border = borda
            if fill:
                c.fill = fill
            if fld == "maps_link" and valor:
                c.hyperlink = valor
                c.value = "📍 Abrir"
                c.font = Font(color="0563C1", underline="single")

    for i, larg in enumerate(LARGURAS, 1):
        ws.column_dimensions[get_column_letter(i)].width = larg

    # Aba Info
    wm = wb.create_sheet("Info")
    wm["A1"], wm["B1"] = "Gerado em:", datetime.now().strftime("%d/%m/%Y %H:%M:%S")
    wm["A2"], wm["B2"] = "Fonte:", "https://servicos.rbmlq.gov.br/dados-abertos/mg/medidores.json"
    wm["A3"], wm["B3"] = "Janela:", f"Últimos {DIAS_RECENTES} dias"
    wm["A4"], wm["B4"] = "Total encontrado:", len(radares)
    wm["A5"], wm["B5"] = "Com coordenadas:", geo_ok

    wb.save(PLANILHA_PATH)
    print(f"\n💾 Planilha salva em:\n   {PLANILHA_PATH}")
    print(f"   {len(radares)} radar(es) | {geo_ok} geocodificado(s)")